# Logic Expert: verification reruns (Colab)

Runs the paper's rerun set, one job at a time on a single GPU (RTX PRO 6000 96 GB or H100 80 GB):
- **Corrected ProofWriter**: 30 runs (rulebase restored, fixed Baseline pooling).
- **MNLI**: 9 runs (loader unaffected by the bugs).

Needed:
- Colab secrets `HF_TOKEN` (with Llama-3.1-8B access) and `WANDB_API_KEY`.
- **Runtime -> Run all.**

Results go to `MyDrive/logic-verify/out/<job>.json` and logs to `MyDrive/logic-verify/logs/`.
The run is resumable: finished jobs are skipped, so after a disconnect just Run all again.
See `logic/docs/verification/VERIFICATION.md` for the protocol.

In [ ]:
import os, subprocess, sys
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['WANDB_ENTITY'] = 'ancorro-oregon-state-university'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:256'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/logic-verify'
os.makedirs(f'{BASE}/out', exist_ok=True); os.makedirs(f'{BASE}/logs', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = '/content/logic-expert-public'
if not os.path.isdir(REPO):
    !git clone -q -b verify-results https://github.com/Ancorro/logic-expert-public.git {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log -1 --oneline
# The pinned eval notebooks locate the repo root through train/train.py.
os.makedirs(f'{REPO}/train', exist_ok=True)
open(f'{REPO}/train/train.py', 'a').close()
!pip -q install "transformers==5.0.0" "wandb==0.25.1" datasets accelerate

In [ ]:
V = f'{REPO}/logic/scripts/verify'
jobs = [l.split() for l in open(f'{V}/manifest/jobs.txt') if l.strip()]
keep = [int(i) for i in open(f'{V}/manifest/paper_indices.txt').read().strip().split(',')]
todo = [jobs[i] for i in keep]
# Order: MNLI seed-42 reproduction check first, then one corrected run, then the rest.
first = ['mnli_inter_s42', 'fix_nogate_s42', 'fix_inter8_s42']
todo.sort(key=lambda j: (first.index(j[0]) if j[0] in first else len(first)))
done = [j[0] for j in todo if os.path.exists(f'{BASE}/out/{j[0]}.json')]
print(len(todo), 'jobs;', len(done), 'already done')

In [ ]:
import time
for name, nb, _tree in todo:
    out = f'{BASE}/out/{name}.json'
    if os.path.exists(out):
        continue
    env = dict(os.environ, VERIFY_OVERRIDES=f'{V}/manifest/{name}.json', VERIFY_OUT=out)
    log = f'{BASE}/logs/{name}.log'
    t0 = time.time()
    print(f'=== {name} ({nb}) started {time.strftime("%H:%M:%S")}', flush=True)
    with open(log, 'w') as fh:
        rc = subprocess.run([sys.executable, '-u', f'{V}/gen/{nb}.py'], cwd=REPO, env=env,
                            stdout=fh, stderr=subprocess.STDOUT).returncode
    status = 'ok' if rc == 0 and os.path.exists(out) else f'FAILED rc={rc} (see {log})'
    print(f'    {name}: {status} in {(time.time() - t0) / 60:.1f} min', flush=True)
    !grep -a -E "epoch=[0-9]+ " {log} | tail -4